## Imports

In [1]:
import sqlite3
import glob
import time
import itertools 

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

/var/folders/m8/kq3tbvsx79n47wprd2zwv5hm0000gn/T/ipykernel_16965/1025221560.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


_____

## 1. Data Ingestion 

Read the data from the database.

In [2]:
annotated_dbs = glob.glob(f'./dataset/fang_system/*.json')

print("Found the following dbs : " , annotated_dbs)

synchronous_annotators = ['group_1' , 'group_2'] 
asynchronous_annotators = ['async_1' , 'async_2' , 'async_3'] 
annotators = synchronous_annotators + asynchronous_annotators
annotator2df = {}

for db_path in  annotated_dbs : 
    merged_df = pd.read_json(db_path)

    for key in annotators : 
        if key in db_path : 
            annotator2df[key] = merged_df

Found the following dbs :  ['./dataset/fang_system/async_1.json', './dataset/fang_system/async_1_selected.json', './dataset/fang_system/group_1.json', './dataset/fang_system/group_2.json', './dataset/fang_system/async_2.json', './dataset/fang_system/async_3.json']


### 1.a. Random Sampling rows for manual annotations

Pull 200 samples from both synchronous and asynchronous groups to manually review label quality. Pulling from top 25th percentile is done later.

In [3]:
synchronous_random_samples = []
asynchronous_random_samples = []

for annotator , df in annotator2df.items(): 
    if annotator in synchronous_annotators: 
        synchronous_random_samples.append(df.sample(n=100))
    if annotator in asynchronous_annotators: 
        asynchronous_random_samples.append(df.sample(n=67))

res_df = pd.concat(synchronous_random_samples)
async_df = pd.concat(asynchronous_random_samples)

res_df = res_df[['text' , 'name']].sample(frac=1)
async_df = async_df[['text' , 'name']].sample(frac=1)

# res_df.to_csv('./dataset/generated_samples/fang_sync_sample_all.csv' , index=False, sep='\t')
# async_df.to_csv('./dataset/generated_samples/fang_async_sample_all.csv' , index=False, sep='\t')

_____

## 2. Jaccard Similarity 

Jaccard similarity for two themes is calculated by the union of their documents divided by the intersection of their documents.

In [4]:
results = []

for anno_1 , anno_2 in itertools.permutations(annotators , 2): 
    anno_1_themes = annotator2df[anno_1]['name'].unique()
    anno_2_themes = annotator2df[anno_2]['name'].unique()
    for anno_1_theme , anno_2_theme in itertools.product(anno_1_themes , anno_2_themes): 
        result = {'anno_1' : anno_1 , 
                  'anno_2' : anno_2 , 
                  'anno_1_theme' : anno_1_theme ,
                  'anno_2_theme' : anno_2_theme}
        anno_1_tweet_ids = set(annotator2df[anno_1][annotator2df[anno_1]['name']==anno_1_theme]['tweet_id'])
        anno_2_tweet_ids = set(annotator2df[anno_2][annotator2df[anno_2]['name']==anno_2_theme]['tweet_id'])
        intersection = anno_1_tweet_ids.intersection(anno_2_tweet_ids)
        union = anno_1_tweet_ids.union(anno_2_tweet_ids)
        jaccard_sim = len(intersection) / len(union)
        result['jaccard_sim'] = jaccard_sim
        results.append(result)

jaccard_df = pd.DataFrame(results)

### 2.a. Getting max jaccard similarity for synchronous experiments

In [5]:
max_jacc_sims = []
filtered_df = jaccard_df[
    (jaccard_df["anno_1"].isin(synchronous_annotators))
    & (jaccard_df["anno_2"].isin(synchronous_annotators))
]

for anno_1_theme in filtered_df["anno_1_theme"].unique():
    if (
        ("kmeans" not in anno_1_theme.lower())
        and ("Unexplainable" not in anno_1_theme.strip())
        and ("topic" not in anno_1_theme.strip())
        and ("Unknown" not in anno_1_theme.strip())
    ):
        theme_filtered_df = filtered_df[
            (filtered_df["anno_1_theme"] == anno_1_theme)
            & ~(filtered_df["anno_2_theme"].str.contains("Kmeans"))
            & ~(filtered_df["anno_2_theme"].str.contains("Unexplainable"))
            & ~(filtered_df["anno_2_theme"].str.contains("topic"))
            & ~(filtered_df["anno_2_theme"].str.contains("Unknown"))
        ]
        max_jacc_sims.append(
            theme_filtered_df.loc[(theme_filtered_df["jaccard_sim"].idxmax())].to_dict()
        )
res_df = pd.DataFrame(max_jacc_sims)

print("Synchronous Jaccard Similarity")
print(f"Average Max Jaccard Similarity: {res_df['jaccard_sim'].mean():.2f}")
print(f"Standard Deviation of Jaccard Similarity: {res_df['jaccard_sim'].std():.2f}")
print(res_df.count())

Synchronous Jaccard Similarity
Average Max Jaccard Similarity: 0.37
Standard Deviation of Jaccard Similarity: 0.31
anno_1          29
anno_2          29
anno_1_theme    29
anno_2_theme    29
jaccard_sim     29
dtype: int64


In [6]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,jaccard_sim
0,group_1,group_2,AntiGunBan,gun rights,0.422222
1,group_1,group_2,EnvironmentalConservation,climate action,0.562862
2,group_1,group_2,SchoolBoardFinances/GasInfrastructure,inflation,0.200929
3,group_1,group_2,AntiGridDeregulation/PoliticalActionAds,gun rights,0.068966
4,group_1,group_2,CleanEnergyGoals,renewables,0.672467
5,group_1,group_2,BellbrookSugarcreekConservativeElections,anti-woke,0.675926
6,group_1,group_2,ConservativePoliticsEvents,indigenous activism,0.240506
7,group_1,group_2,TPUSAAd,TPUSA,0.821429
8,group_1,group_2,DomesticOilAndGasProduction,inflation,0.056522
9,group_1,group_2,ClimateChange,climate action,0.060784


### 2.b. Getting max jaccard similarity for asynchronous experiments

In [7]:
max_jacc_sims = []
filtered_df = jaccard_df[
    (jaccard_df["anno_1"].isin(asynchronous_annotators))
    & (jaccard_df["anno_2"].isin(asynchronous_annotators))
]
for anno_1_theme in filtered_df["anno_1_theme"].unique():
    if (
        ("kmeans" not in anno_1_theme.lower())
        and ("Unexplainable" not in anno_1_theme.strip())
        and ("topic" not in anno_1_theme.strip())
        and ("Unknown" not in anno_1_theme.strip())
    ):
        theme_filtered_df = filtered_df[
            (filtered_df["anno_1_theme"] == anno_1_theme)
            & ~(filtered_df["anno_2_theme"].str.contains("Kmeans"))
            & ~(filtered_df["anno_2_theme"].str.contains("Unexplainable"))
            & ~(filtered_df["anno_2_theme"].str.contains("topic"))
            & ~(filtered_df["anno_2_theme"].str.contains("Unknown"))
        ]
        max_jacc_sims.append(
            theme_filtered_df.loc[(theme_filtered_df["jaccard_sim"].idxmax())].to_dict()
        )
res_df = pd.DataFrame(max_jacc_sims)

print("Asynchronous Jaccard Similarity")
print(f"Average Max Jaccard Similarity: {res_df['jaccard_sim'].mean():.2f}")
print(f"Standard Deviation of Jaccard Similarity: {res_df['jaccard_sim'].std():.2f}")

Asynchronous Jaccard Similarity
Average Max Jaccard Similarity: 0.58
Standard Deviation of Jaccard Similarity: 0.29


In [8]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,jaccard_sim
0,async_3,async_2,AntiGunBan,pro-2A,0.581395
1,async_1,async_3,AntiWoke,AntiLiberalElectionCandidates,0.695035
2,async_1,async_3,ProtectEnvironment,SaveEnvironment,0.724792
3,async_1,async_3,CleanEnergyWorks,ProRenewableEnergy,0.656859
4,async_1,async_2,ActionAgainstDemSpending,criticism of liberal tax policy,0.767528
5,async_1,async_3,IndigenousAndBlackAdvocacy,MinorityGroupWomenAdvocacy,0.673469
6,async_1,async_2,PassBidenAct,promoting legislation against climate change,0.834116
7,async_1,async_3,AntiOilPolitician,ProCleanEnergyPolitician,0.754601
8,async_1,async_2,AntiPropertyTax,discussing problems with school board,0.875000
9,async_1,async_2,ProUSOilIndustry,criticism for disfavoring US ONG production,0.574468


______

## 3. Centroid Cosine Similarity 

### 3.a. Loading SBERT Vectors 

In [9]:
sbert_vectors = np.load('./dataset/sbert.npy')

### 3.b. Calculating centroids for sync + async experiments

In [10]:
results = []

for annotator, df in annotator2df.items():
    cosine_sims = []
    themes = []
    for i, theme in enumerate(df["name"].unique()):
        if (
            ("kmeans" not in theme.lower())
            and ("Unexplainable" not in theme.strip())
            and ("topic" not in theme.strip())
            and ("Unknown" not in theme.strip())
        ):
            result = {"annotator": annotator}
            ids = df[df["name"] == theme]["tweet_id"].tolist()
            result["theme"] = theme
            theme_vectors = sbert_vectors[ids]
            theme_centroid = np.average(theme_vectors, axis=0, keepdims=True)

            result["theme_centroid"] = theme_centroid
            result["theme_vectors"] = theme_vectors

            dot_prod = np.dot(theme_centroid, theme_vectors.T).squeeze(0)
            dot_prod = np.expand_dims(dot_prod, axis=-1)

            norm = np.linalg.norm(theme_vectors, axis=1, keepdims=True)
            cosine_sim = dot_prod / norm
            result["cosine_sim"] = cosine_sim
            results.append(result)

### 3.c. Calculating synchronous centroid cosine similarity

In [11]:
cosine_sim_results = []

for anno_1, anno_2 in itertools.permutations(synchronous_annotators, 2):
    anno_1_results = [r for r in results if r["annotator"] == anno_1]
    anno_2_results = [r for r in results if r["annotator"] == anno_2]
    for anno_1_result in anno_1_results:
        for anno_2_result in anno_2_results:
            if (
                ("kmeans" not in anno_1_result["theme"].lower())
                and ("Unexplainable" not in anno_1_result["theme"].strip())
                and ("topic" not in anno_1_result["theme"].strip())
                and ("Unknown" not in anno_1_result["theme"].strip())
                and ("kmeans" not in anno_2_result["theme"].lower())
                and ("Unexplainable" not in anno_2_result["theme"].strip())
                and ("topic" not in anno_2_result["theme"].strip())
                and ("Unknown" not in anno_2_result["theme"].strip())
            ):
                cosine_sim = cosine_similarity(
                    anno_1_result["theme_centroid"], anno_2_result["theme_centroid"]
                )
                cosine_sim_result = {
                    "anno_1": anno_1,
                    "anno_2": anno_2,
                    "anno_1_theme": anno_1_result["theme"],
                    "anno_2_theme": anno_2_result["theme"],
                    "cosine_sim": cosine_sim.squeeze(),
                }
                cosine_sim_results.append(cosine_sim_result)

cosine_sim_df = pd.DataFrame(cosine_sim_results)

In [12]:
max_cosine_sims = []

for anno_1, anno_2 in itertools.permutations(synchronous_annotators, 2):
    filtered_df = cosine_sim_df[
        (cosine_sim_df["anno_1"] == anno_1) & (cosine_sim_df["anno_2"] == anno_2)
    ]
    for anno_1_theme in filtered_df["anno_1_theme"].unique():
        if (
            ("kmeans" not in anno_1_result["theme"].lower())
            and ("Unexplainable" not in anno_1_result["theme"].strip())
            and ("topic" not in anno_1_result["theme"].strip())
            and ("Unknown" not in anno_1_result["theme"].strip())
            and ("kmeans" not in anno_2_result["theme"].lower())
            and ("Unexplainable" not in anno_2_result["theme"].strip())
            and ("topic" not in anno_2_result["theme"].strip())
            and ("Unknown" not in anno_2_result["theme"].strip())
        ):
            theme_filtered_df = filtered_df[
                (filtered_df["anno_1_theme"] == anno_1_theme)
                & ~(filtered_df["anno_2_theme"].str.contains("Kmeans"))
                & ~(filtered_df["anno_2_theme"].str.contains("Unknown"))
            ]
            max_cosine_sims.append(
                theme_filtered_df.loc[
                    (theme_filtered_df["cosine_sim"].idxmax())
                ].to_dict()
            )

res_df = pd.DataFrame(max_cosine_sims)


print("Synchronous Centroid Cosine Similarity")
print(f"Average Max Centroid Cosine Similarity: {res_df['cosine_sim'].mean():.2f}")
print(
    f"Standard Deviation of Centroid Cosine Similarity: {res_df['cosine_sim'].std():.2f}"
)
print(res_df.count())

Synchronous Centroid Cosine Similarity
Average Max Centroid Cosine Similarity: 0.89
Standard Deviation of Centroid Cosine Similarity: 0.14
anno_1          29
anno_2          29
anno_1_theme    29
anno_2_theme    29
cosine_sim      29
dtype: int64


In [13]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,cosine_sim
0,group_1,group_2,AntiGunBan,gun rights,0.97422516
1,group_1,group_2,EnvironmentalConservation,climate action,0.9917236
2,group_1,group_2,SchoolBoardFinances/GasInfrastructure,inflation,0.9255009
3,group_1,group_2,AntiGridDeregulation/PoliticalActionAds,gun rights,0.80143815
4,group_1,group_2,CleanEnergyGoals,renewables,0.99667275
5,group_1,group_2,BellbrookSugarcreekConservativeElections,anti-woke,0.9893303
6,group_1,group_2,ConservativePoliticsEvents,indigenous activism,0.9292284
7,group_1,group_2,TPUSAAd,TPUSA,0.99082446
8,group_1,group_2,DomesticOilAndGasProduction,inflation,0.89372194
9,group_1,group_2,ClimateChange,climate action,0.8998476


### 3.d. Calculating asynchronous centroid cosine similarity

In [14]:
cosine_sim_results = []

for anno_1, anno_2 in itertools.permutations(asynchronous_annotators, 2):
    anno_1_results = [r for r in results if r["annotator"] == anno_1]
    anno_2_results = [r for r in results if r["annotator"] == anno_2]
    for anno_1_result in anno_1_results:
        for anno_2_result in anno_2_results:
            if (
                ("kmeans" not in anno_1_result["theme"].lower())
                and ("Unexplainable" not in anno_1_result["theme"].strip())
                and ("topic" not in anno_1_result["theme"].strip())
                and ("Unknown" not in anno_1_result["theme"].strip())
                and ("kmeans" not in anno_2_result["theme"].lower())
                and ("Unexplainable" not in anno_2_result["theme"].strip())
                and ("topic" not in anno_2_result["theme"].strip())
                and ("Unknown" not in anno_2_result["theme"].strip())
            ):
                cosine_sim = cosine_similarity(
                    anno_1_result["theme_centroid"], anno_2_result["theme_centroid"]
                )
                cosine_sim_result = {
                    "anno_1": anno_1,
                    "anno_2": anno_2,
                    "anno_1_theme": anno_1_result["theme"],
                    "anno_2_theme": anno_2_result["theme"],
                    "cosine_sim": cosine_sim.squeeze(),
                }
                cosine_sim_results.append(cosine_sim_result)

cosine_sim_df = pd.DataFrame(cosine_sim_results)

In [15]:
max_cosine_sims = []

for anno_1, anno_2 in itertools.permutations(asynchronous_annotators, 2):
    filtered_df = cosine_sim_df[
        (cosine_sim_df["anno_1"] == anno_1) & (cosine_sim_df["anno_2"] == anno_2)
    ]
    for anno_1_theme in filtered_df["anno_1_theme"].unique():
        if (
            ("kmeans" not in anno_1_theme.lower())
            and ("unexplainable" not in anno_1_theme.lower())
            and ("topic" not in anno_1_theme.lower())
            and ("unknown" not in anno_1_theme.lower())
        ):
            theme_filtered_df = filtered_df[
                (filtered_df["anno_1_theme"] == anno_1_theme)
                & ~(filtered_df["anno_2_theme"].str.contains("Kmeans"))
                & ~(filtered_df["anno_2_theme"].str.contains("Unknown"))
            ]
            max_cosine_sims.append(
                theme_filtered_df.loc[
                    (theme_filtered_df["cosine_sim"].idxmax())
                ].to_dict()
            )

res_df = pd.DataFrame(max_cosine_sims)

print("Asynchronous Centroid Cosine Similarity")
print(f"Average Max Centroid Cosine Similarity: {res_df['cosine_sim'].mean():.2f}")
print(
    f"Standard Deviation of Centroid Cosine Similarity: {res_df['cosine_sim'].std():.2f}"
)
print(res_df.count())

Asynchronous Centroid Cosine Similarity
Average Max Centroid Cosine Similarity: 0.93
Standard Deviation of Centroid Cosine Similarity: 0.12
anno_1          92
anno_2          92
anno_1_theme    92
anno_2_theme    92
cosine_sim      92
dtype: int64


In [16]:
# To view individual similarities
res_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,cosine_sim
0,async_1,async_2,AntiGunBan,pro-2A,0.9608255
1,async_1,async_2,AntiWoke,advocating for city level politics,0.9899468
2,async_1,async_2,ProtectEnvironment,donations for climate change,0.9940119
3,async_1,async_2,CleanEnergyWorks,pros of wind and solar energy,0.9248527
4,async_1,async_2,ActionAgainstDemSpending,criticism of liberal tax policy,0.99899626
...,...,...,...,...,...
87,async_3,async_2,ProCleanEnergyPolitician,kudos for confronting ONG emissions,0.9884269
88,async_3,async_2,TPUSALiveAd,promoting news and oped program,0.9919358
89,async_3,async_2,AntiLiberalSpending,criticism of liberal tax policy,0.8940756
90,async_3,async_2,AntiPropertyTax,discussing problems with school board,0.8627924


____

## 4. Group Average Cosine Similarity

In [17]:
results = {}
annotators = []

for annotator, df in annotator2df.items():
    results[annotator] = {}
    annotators.append(annotator)
    for i, theme in enumerate(df["name"].unique()):
        if (
            ("kmeans" not in theme.lower())
            and ("Unexplainable" not in theme.strip())
            and ("topic" not in theme.strip())
            and ("Unknown" not in theme.strip())
        ):
            ids = df[df["name"] == theme]["tweet_id"].tolist()
            theme_vectors = sbert_vectors[ids]
            results[annotator][theme] = theme_vectors

### 4.a. Calculating Group Average Similarities

In [18]:
global_average_sims = []
total_combos = len(list(itertools.combinations_with_replacement(annotators, 2)))

for ctr , (anno1 , anno2) in enumerate(itertools.combinations_with_replacement(annotators, 2)):
  anno1_themes = results[anno1]
  anno2_themes = results[anno2]
  for anno1_theme, anno1_theme_vectors in anno1_themes.items():
    for anno2_theme, anno2_theme_vectors in anno2_themes.items():
      s_time = time.time()
      cosine_sims = cosine_similarity(anno1_theme_vectors , anno2_theme_vectors)
      average_sim = np.average(cosine_sims)
      std_deviation = np.std(cosine_sims)
      global_average_sims.append({'anno1' : anno1 ,
                                  'anno2' : anno2 ,
                                  'anno1_theme' : anno1_theme ,
                                  'anno2_theme' : anno2_theme ,
                                  'average_sim' : average_sim ,
                                  'std_deviation' : std_deviation
                                  })
      print(f"Completed {ctr+1}/{total_combos} combinations.\nTotal time to calc : {time.time() - s_time} seconds. \n{global_average_sims[-1]}\n--------\n")
      # save for reference since this calculation runs much slower than for pacheco data
      pd.DataFrame(global_average_sims).to_csv('./dataset/fang_system/std_dev_aggregated_sims.csv', index=False)

Completed 1/15 combinations.
Total time to calc : 0.002836942672729492 seconds. 
{'anno1': 'async_1', 'anno2': 'async_1', 'anno1_theme': 'AntiGunBan', 'anno2_theme': 'AntiGunBan', 'average_sim': 0.303288, 'std_deviation': 0.13467766}
--------

Completed 1/15 combinations.
Total time to calc : 0.0022346973419189453 seconds. 
{'anno1': 'async_1', 'anno2': 'async_1', 'anno1_theme': 'AntiGunBan', 'anno2_theme': 'AntiWoke', 'average_sim': 0.27441308, 'std_deviation': 0.10325976}
--------

Completed 1/15 combinations.
Total time to calc : 0.010207891464233398 seconds. 
{'anno1': 'async_1', 'anno2': 'async_1', 'anno1_theme': 'AntiGunBan', 'anno2_theme': 'ProtectEnvironment', 'average_sim': 0.23161182, 'std_deviation': 0.11290237}
--------

Completed 1/15 combinations.
Total time to calc : 0.015947818756103516 seconds. 
{'anno1': 'async_1', 'anno2': 'async_1', 'anno1_theme': 'AntiGunBan', 'anno2_theme': 'CleanEnergyWorks', 'average_sim': 0.20948875, 'std_deviation': 0.120016314}
--------

Comp

KeyboardInterrupt: 

In [ ]:
# Read from csv
# global_average_sims = pd.read_csv('./dataset/fang_system/std_dev_aggregated_sims.csv')

In [ ]:
global_average_sims

[{'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AntiGunBan',
  'anno2_theme': 'AntiGunBan',
  'average_sim': 0.303288,
  'std_deviation': 0.13467766},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AntiGunBan',
  'anno2_theme': 'AntiWoke',
  'average_sim': 0.27441308,
  'std_deviation': 0.10325976},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AntiGunBan',
  'anno2_theme': 'ProtectEnvironment',
  'average_sim': 0.23161182,
  'std_deviation': 0.11290237},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AntiGunBan',
  'anno2_theme': 'CleanEnergyWorks',
  'average_sim': 0.20948875,
  'std_deviation': 0.120016314},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AntiGunBan',
  'anno2_theme': 'ActionAgainstDemSpending',
  'average_sim': 0.24456517,
  'std_deviation': 0.11794432},
 {'anno1': 'async_1',
  'anno2': 'async_1',
  'anno1_theme': 'AntiGunBan',
  'anno2_theme': 'IndigenousAndBlackAdvocacy',
  'average_sim': 0

In [ ]:
global_average_df = pd.DataFrame(global_average_sims)

### 4.b. Calculating Synchronous Group Average Cosine Similarities

In [ ]:
max_cosine_sims = []

for anno_1, anno_2 in itertools.permutations(synchronous_annotators, 2):
    filtered_df = global_average_df[
        (global_average_df["anno1"] == anno_1) & (global_average_df["anno2"] == anno_2)
    ]
    for anno_1_theme in filtered_df["anno1_theme"].unique():
        if (
            ("kmeans" not in anno_1_theme.lower())
            and ("unexplainable" not in anno_1_theme.lower())
            and ("topic" not in anno_1_theme.lower())
            and ("unknown" not in anno_1_theme.lower())
        ):
            theme_filtered_df = filtered_df[
                (filtered_df["anno1_theme"] == anno_1_theme)
                & ~(filtered_df["anno2_theme"].str.contains("Kmeans"))
                & ~(filtered_df["anno2_theme"].str.contains("Unknown"))
            ]
            max_cosine_sims.append(
                theme_filtered_df.loc[
                    (theme_filtered_df["average_sim"].idxmax())
                ].to_dict()
            )

res_df = pd.DataFrame(max_cosine_sims)

print("Synchronous Group Average Cosine Similarity")
print(f"Average Max Group Cosine Similarity: {res_df['average_sim'].mean():.2f}")
print(
    f"Standard Deviation of Group Cosine Similarity: { res_df['average_sim'].std():.2f}"
)

print(res_df.count())

Synchronous Group Average Cosine Similarity
Average Max Group Cosine Similarity: 0.43
Standard Deviation of Group Cosine Similarity: 0.14
anno1            19
anno2            19
anno1_theme      19
anno2_theme      19
average_sim      19
std_deviation    19
dtype: int64


### 4.c. Calculating Asynchronous Group Average Cosine Similarities

In [ ]:
max_cosine_sims = []

for anno_1, anno_2 in itertools.permutations(asynchronous_annotators, 2):
    filtered_df = global_average_df[
        (global_average_df["anno1"] == anno_1) & (global_average_df["anno2"] == anno_2)
    ]
    for anno_1_theme in filtered_df["anno1_theme"].unique():
        if (
            ("kmeans" not in anno_1_theme.lower())
            and ("unexplainable" not in anno_1_theme.lower())
            and ("topic" not in anno_1_theme.lower())
            and ("unknown" not in anno_1_theme.lower())
        ):
            theme_filtered_df = filtered_df[
                (filtered_df["anno1_theme"] == anno_1_theme)
                & ~(filtered_df["anno2_theme"].str.contains("Kmeans"))
                & ~(filtered_df["anno2_theme"].str.contains("Unknown"))
            ]
            max_cosine_sims.append(
                theme_filtered_df.loc[
                    (theme_filtered_df["average_sim"].idxmax())
                ].to_dict()
            )

res_df = pd.DataFrame(max_cosine_sims)


print("Asynchronous Group Average Cosine Similarity")
print(f"Average Max Group Cosine Similarity: {res_df['average_sim'].mean():.2f}")
print(
    f"Standard Deviation of Group Cosine Similarity: { res_df['average_sim'].std():.2f}"
)

print(res_df.count())

Asynchronous Group Average Cosine Similarity
Average Max Group Cosine Similarity: 0.46
Standard Deviation of Group Cosine Similarity: 0.16
anno1            42
anno2            42
anno1_theme      42
anno2_theme      42
average_sim      42
std_deviation    42
dtype: int64


____

## 5. Getting weights by percentile buckets

In [ ]:
results = []

for annotator, df in annotator2df.items():
    cosine_sims = []
    themes = []
    for i, theme in enumerate(df["name"].unique()):
        if (
            ("kmeans" not in theme.lower())
            and ("unexplainable" not in theme.lower())
            and ("topic" not in theme.lower())
            and ("unknown" not in theme.lower())
        ):
            result = {"annotator": annotator}
            ids = df[df["name"] == theme]["tweet_id"].tolist()
            result["theme"] = theme
            result["tweets"] = df[df["name"] == theme]["text"].tolist()
            result["weights"] = df[df["name"] == theme]["weight"].tolist()

            theme_vectors = sbert_vectors[ids]
            theme_centroid = np.average(theme_vectors, axis=0, keepdims=True)

            result["theme_centroid"] = theme_centroid
            result["theme_vectors"] = theme_vectors

            dot_prod = np.dot(theme_centroid, theme_vectors.T).squeeze(0)
            dot_prod = np.expand_dims(dot_prod, axis=-1)

            norm = np.linalg.norm(theme_vectors, axis=1, keepdims=True)
            cosine_sim = dot_prod / norm
            result["cosine_sim"] = cosine_sim.squeeze()
            results.append(result)

In [ ]:
quartile_map = {
    "0-25": (0, 25),
    "25-50": (25, 50),
    "50-75": (50, 75),
    "75-100": (75, 100),
}

quartile_results = {
    quartile: {'tweets': [],
              'vectors': [],
              'theme': [],
              'annotator': []}
    for quartile in quartile_map.keys()
}

for result in results:
    for quartile, (n, m) in quartile_map.items():
        low = np.percentile(result["weights"], n)
        high = np.percentile(result["weights"], m)
        
        quartile_indices = np.where((result["weights"] >= low) & (result["weights"] <= high))
        quartile_vectors = result["theme_vectors"][quartile_indices]
        quartile_tweets = np.array(result["tweets"])[quartile_indices]
        
        annotator = result["annotator"]
        theme = result["theme"]

        quartile_results[quartile]["tweets"].extend(quartile_tweets)
        quartile_results[quartile]["vectors"].extend(quartile_vectors)
        quartile_results[quartile]["theme"].extend([theme] * quartile_tweets.shape[0])
        quartile_results[quartile]["annotator"].extend([annotator] * quartile_tweets.shape[0])

quartile_dfs = {
    quartile: pd.DataFrame(data=quartile_data, columns=["theme", "annotator", "tweets"])
    for quartile, quartile_data in quartile_results.items()
}

    # m = np.percentile(result["weights"], 75)
    # top_25_indices = np.where(result["weights"] > m)
    # top_25_vectors = result["theme_vectors"][top_25_indices]
    # top_25_tweets = np.array(result["tweets"])[top_25_indices]
    # annotator = result["annotator"]
    # theme = result["theme"]

    # top_25_results["top_25_tweets"].extend(top_25_tweets)
    # top_25_results["top_25_vectors"].extend(top_25_vectors)
    # top_25_results["theme"].extend([theme] * top_25_tweets.shape[0])
    # top_25_results["annotator"].extend([annotator] * top_25_tweets.shape[0])


# top_25_df = pd.DataFrame(
#     top_25_results, columns=["theme", "annotator", "top_25_tweets"]
# )

### 4.a. Random Sampling rows for manual annotations

In [ ]:
for quartile, df in quartile_dfs.items():
    # Get unique themes in this quartile
    # themes = df['theme'].unique()
    async_themes = df[df['annotator'].isin(asynchronous_annotators)]['theme'].unique()
    sync_themes = df[df['annotator'].isin(synchronous_annotators)]['theme'].unique()
    
    # Calculate samples per theme (200 total / number of themes)
    # samples_per_theme = 200 // len(themes)
    # remaining_samples = 200 % len(themes)
    async_samples_per_theme = 50 // len(async_themes)
    async_remaining_samples = 50 % len(async_themes)
    sync_samples_per_theme = 52 // len(sync_themes)
    sync_remaining_samples = 52 % len(sync_themes)
    
    # Sample from synchronous annotators
    sync_samples = []
    for i, theme in enumerate(sync_themes):
        theme_df = df[(df['annotator'].isin(synchronous_annotators)) & (df['theme'] == theme)]
        n_samples = sync_samples_per_theme + (1 if i < sync_remaining_samples else 0)
        n_samples = min(n_samples, len(theme_df))  # Don't sample more than available
        if n_samples > 0:
            sync_samples.append(theme_df.sample(n=n_samples))
    
    sync_quartile = pd.concat(sync_samples, ignore_index=True)[['theme', 'tweets']]
    
    # Sample from asynchronous annotators
    async_samples = []
    for i, theme in enumerate(async_themes):
        theme_df = df[(df['annotator'].isin(asynchronous_annotators)) & (df['theme'] == theme)]
        n_samples = async_samples_per_theme + (1 if i < async_remaining_samples else 0)
        n_samples = min(n_samples, len(theme_df))  # Don't sample more than available
        if n_samples > 0:
            async_samples.append(theme_df.sample(n=n_samples))
    
    async_quartile = pd.concat(async_samples, ignore_index=True)[['theme', 'tweets']]
    
    # Save to CSV
    sync_quartile.to_csv(f'./dataset/generated_samples/fang_sync_sample_{quartile}_new.csv', index=False, sep='\t')
    async_quartile.to_csv(f'./dataset/generated_samples/fang_async_sample_{quartile}_new.csv', index=False, sep='\t')
    print(len(sync_quartile), len(async_quartile))


51 50
50 50
51 50
51 50


____

## 5. Calculating Intra- and Inter-cluster Similarity

In [ ]:
top_25_results = {}

for result in results: 

    top_25_results[result['theme']] = {}

    m = np.percentile(result['weights'] , 75)
    top_25_indices = np.where(result['weights']>=m)
    top_25_vectors = result['theme_vectors'][top_25_indices]
    top_25_tweets = np.array(result['tweets'])[top_25_indices]
    annotator=result['annotator']
    theme=result['theme']

    top_25_results[theme]['top_25_tweets'] = (top_25_tweets)
    top_25_results[theme]['top_25_vectors'] = (top_25_vectors )
    top_25_results[theme]['annotator'] = (annotator)

### 5.a. Calculating similarities for top 25th percentile clusters

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

top_25_average_sims = []

for anno_1_theme , anno1_top_25 in tqdm(top_25_results.items()): 
    for anno_2_theme , anno2_top_25 in top_25_results.items(): 

        anno1_top_25_vectors = anno1_top_25['top_25_vectors']
        anno2_top_25_vectors = anno2_top_25['top_25_vectors']

        cosine_sims = cosine_similarity(anno1_top_25_vectors , anno2_top_25_vectors)
        average_sim = np.average(cosine_sims)
        std_deviation = np.std(cosine_sims)

        top_25_average_sims.append({'anno_1' : anno1_top_25['annotator'] , 
                                   'anno_2' : anno2_top_25['annotator'], 
                                   'anno_1_theme' : anno_1_theme,
                                   'anno_2_theme' : anno_2_theme,
                                   'average_sim' : average_sim , 
                                   'std_dev_sim' : std_deviation})


  0%|          | 0/71 [00:00<?, ?it/s]

In [ ]:
top_25_average_df = pd.DataFrame(top_25_average_sims)
top_25_average_df

,anno_1,anno_2,anno_1_theme,anno_2_theme,average_sim,std_dev_sim
0,async_3,async_3,AntiGunBan,AntiGunBan,0.596259,0.171514
1,async_3,async_1,AntiGunBan,AntiWoke,0.386459,0.090362
2,async_3,async_1,AntiGunBan,ProtectEnvironment,0.237373,0.094159
3,async_3,async_1,AntiGunBan,CleanEnergyWorks,0.143717,0.076706
4,async_3,async_1,AntiGunBan,ActionAgainstDemSpending,0.244062,0.090942
...,...,...,...,...,...,...
5036,async_3,async_3,EnergyInfrastructureProjects,WaterPollutionAndContamination,0.182567,0.096527
5037,async_3,async_3,EnergyInfrastructureProjects,ProCleanEnergyPolitician,0.197540,0.093913
5038,async_3,async_3,EnergyInfrastructureProjects,TPUSALiveAd,0.108311,0.031367
5039,async_3,async_3,EnergyInfrastructureProjects,AntiLiberalSpending,0.093319,0.041961


In [ ]:
top_25_intra_cluster_df = top_25_average_df[(top_25_average_df['anno_1']==top_25_average_df['anno_2']) & (top_25_average_df['anno_1_theme']==top_25_average_df['anno_2_theme'])] 

### 5.b. Intra- and Inter-theme similarities for top 25th percentile subsets in synchronous and asynchronous experiments

In [ ]:
print(f"Synchronous top 25% intra theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous top 25% intra theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous top 25% intra theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous top 25% intra theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous top 25% intra theme similarity average: 0.72
Synchronous top 25% intra theme similarity standard deviation: 0.21
Asynchronous top 25% intra theme similarity average: 0.65
Asynchronous top 25% intra theme similarity standard deviation: 0.20


In [ ]:
top_25_intra_cluster_df = top_25_average_df[(top_25_average_df['anno_1']==top_25_average_df['anno_2']) & (top_25_average_df['anno_1_theme']!=top_25_average_df['anno_2_theme'])] 

In [ ]:
print(f"Synchronous top 25% inter theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous top 25% inter theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous top 25% inter theme similarity average: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous top 25% inter theme similarity standard deviation: {top_25_intra_cluster_df[top_25_intra_cluster_df['anno_1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous top 25% inter theme similarity average: 0.26
Synchronous top 25% inter theme similarity standard deviation: 0.09
Asynchronous top 25% inter theme similarity average: 0.26
Asynchronous top 25% inter theme similarity standard deviation: 0.08


### 5.c. Intra- and Inter-theme similarities for whole set in synchronous and asynchronous experiments

In [ ]:
global_average_df = pd.DataFrame(global_average_sims)

intra_global_average_df = global_average_df[(global_average_df['anno1']==global_average_df['anno2']) & (global_average_df['anno1_theme']==global_average_df['anno2_theme'])] 


print(f"Synchronous global intra theme similarity average: {intra_global_average_df[intra_global_average_df['anno1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous global intra theme similarity standard deviation: {intra_global_average_df[intra_global_average_df['anno1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous global intra theme similarity average: {intra_global_average_df[intra_global_average_df['anno1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous global intra theme similarity standard deviation: {intra_global_average_df[intra_global_average_df['anno1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous global intra theme similarity average: 0.57
Synchronous global intra theme similarity standard deviation: 0.23
Asynchronous global intra theme similarity average: 0.50
Asynchronous global intra theme similarity standard deviation: 0.19


In [ ]:
inter_global_avg_df = global_average_df[(global_average_df['anno1']==global_average_df['anno2']) & (global_average_df['anno1_theme']!=global_average_df['anno2_theme'])] 

print(f"Synchronous global inter theme similarity average: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(synchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Synchronous global inter theme similarity standard deviation: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(synchronous_annotators)]['average_sim'].std():.2f}")


print(f"Asynchronous global inter theme similarity average: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(asynchronous_annotators)]['average_sim'].mean():.2f}")
print(f"Asynchronous global inter theme similarity standard deviation: {inter_global_avg_df[inter_global_avg_df['anno1'].isin(asynchronous_annotators)]['average_sim'].std():.2f}")

Synchronous global inter theme similarity average: 0.25
Synchronous global inter theme similarity standard deviation: 0.08
Asynchronous global inter theme similarity average: 0.24
Asynchronous global inter theme similarity standard deviation: 0.07
